### First derivative: complex trick

The first derivative of a function can be computed as:

$$
    f'(x) = \lim_{h \rightarrow 0} \frac{\operatorname{Im} \left\{ f(x + i h) \right\} }{h}
$$

Using this formulation, catastrophic cancellation is avoided and roundoff error is reduced. We can reach very high precision 

In [1]:
import numpy as np

In [2]:
def forward(f, x0, h): # Classical
    return (f(x0 + h) - f(x0)) / h

def forward_c(f, x0, h): # Complex definition
    return np.imag(f(x0 + 1j * h)) / h

In [3]:
f = lambda x: np.sqrt(x) * np.exp(- x ** 4) # Target function
x0 = 1.0 # Target point

In [4]:
hs = 10.0 ** np.arange(-1, -16 - 1, -1) # Array of h values: 1e-01, 1e-02, 1e-03, ... , 1e-16

In [5]:
print(np.array([forward(f, x0, h) for h in hs])) # If h is too small or too large, this fails. Catastrophic cancellation effect is macroscopic at h = 1e-16
print(np.array([forward_c(f, x0, h) for h in hs])) # Stable for every value of h

[-1.2530458  -1.28763722 -1.28762002 -1.2875826  -1.2875785  -1.28757809
 -1.28757805 -1.28757804 -1.28757816 -1.28757838 -1.2875756  -1.28769218
 -1.28674849 -1.29340982 -1.49880108  0.        ]
[-1.32854282 -1.28797897 -1.28758205 -1.28757808 -1.28757804 -1.28757804
 -1.28757804 -1.28757804 -1.28757804 -1.28757804 -1.28757804 -1.28757804
 -1.28757804 -1.28757804 -1.28757804 -1.28757804]


In [6]:
def derivative_c(f, x0, h0 = 1.0, eps = 1e-15, max_iter = 100): # Computes the derivative of f(x) in x0 with desired precision
    
    def forward_c(f, x0, h):
        return np.imag(f(x0 + 1j * h)) / h

    h = h0
    for j in range(max_iter):
        
        D1, D2 = forward_c(f, x0, h), forward_c(f, x0, h / 2.0)
        
        errD1 = 4.0 * np.abs(D2 - D1) / 3.0
        if errD1 < eps: # Halt when the difference is below eps 
            break

        h /= 2.0 # Update h

    print(f"Max number of iterations reached. h = {h}") if j == max_iter else print(f"Number of iterations: {j + 1}. h = {h}")
    if errD1 == 0.0: print("Machine precision result")
    
    D1 = float(D1)

    return (D1, errD1, errD1 / np.abs(D1)) if (errD1 != 0.0) else float(D1) # Returns the derivative and the estimated absolute/relative error (Richardson)

In [ ]:
derivative_c(f, x0, eps = 1e-14) # Number of iterations depends on h0 and eps

Number of iterations: 26. h = 2.9802322387695312e-08


(-1.2875780441000515,
 np.float64(3.2566542055671257e-15),
 np.float64(2.5292868424479493e-15))

Result:

$$
    - 1.287578044100051 \pm 0.000000000000003
$$

In [8]:
ex = float(-(np.exp(- x0 ** 4) * (8.0 * x0 ** 4 - 1.0) / (2.0 * np.sqrt(x0)))) # Exact value
print(f"{ex:.15f}")

-1.287578044100048


In [9]:
derivative_c(f, x0, eps = 1e-15) # Machine precision accuracy

Number of iterations: 28. h = 7.450580596923828e-09
Machine precision result


-1.2875780441000482

In [10]:
print(f"{ex:.16f}")

-1.2875780441000482
